# **Spotify Dataset — Data Engineer**

## **Import Libraries**

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

## **Project path Configuration**

In [27]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_data.csv"

## **Load the Dataset**

In [28]:
df = pd.read_csv(RAW_DATA_PATH)

### Basic Dataset Check

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [30]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## **Data Cleaning**

### Remove Unnecessary Index Column

This section removes automatically generated index columns that are not useful for analysis.

In [31]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head(1)

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,acoustic


### **Check Missing Values**
This section handles missing values in the dataset.

- Columns with excessive missing values are removed to maintain data quality.  
- After that, rows containing remaining missing values are dropped to ensure consistency for further analysis.

In [32]:
missing_values = df.isnull().sum()

missing_table = missing_values[missing_values > 0].to_frame(name='Missing Count')

print("Columns with missing values:")
print(missing_table)

Columns with missing values:
            Missing Count
artists                 1
album_name              1
track_name              1


### Remove columns with excessive missing values

Columns with a missing value ratio greater than 40% are removed to improve dataset reliability and reduce noise in further analysis.

In [33]:
col_threshold = 0.4    
df = df.loc[:, df.isnull().mean() < col_threshold]
df.shape

(114000, 20)

### Remove remaining Missing values

After removing low-quality columns, rows containing remaining missing values are dropped to ensure dataset consistency.

In [34]:
df = df.dropna()

### Verify missing values after cleaning

This step verifies that the dataset no longer contains missing values after the cleaning process.

In [35]:
df.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

### **Duplicate**
This section checks duplicated records in the dataset.

In [36]:
duplicated_rows = df.duplicated().sum()

print(f"Number of duplicated rows: {duplicated_rows}")

Number of duplicated rows: 450


In [37]:
df.drop_duplicates(inplace=True) 

In [38]:
df.reset_index(drop=True, inplace=True)

### **Outlier Detection**

This section identifies numerical columns for outlier processing using the IQR method.

In [39]:
numeric_columns = df.select_dtypes(include=np.number).columns
numeric_columns

Index(['popularity', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'],
      dtype='str')

### Remove Outliers Using IQR

In [40]:
for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = (Q1 - 1.5 * IQR)
    upper_bound = (Q3 + 1.5 * IQR)

    df = df[(df[col] >= lower_bound)&(df[col] <= upper_bound)]

df.shape

(59721, 20)

## **Final Dataset Check**

This section verifies the final dataset structure after preprocessing and cleaning.

In [41]:
df.info()

<class 'pandas.DataFrame'>
Index: 59721 entries, 0 to 113548
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          59721 non-null  str    
 1   artists           59721 non-null  str    
 2   album_name        59721 non-null  str    
 3   track_name        59721 non-null  str    
 4   popularity        59721 non-null  int64  
 5   duration_ms       59721 non-null  int64  
 6   explicit          59721 non-null  bool   
 7   danceability      59721 non-null  float64
 8   energy            59721 non-null  float64
 9   key               59721 non-null  int64  
 10  loudness          59721 non-null  float64
 11  mode              59721 non-null  int64  
 12  speechiness       59721 non-null  float64
 13  acousticness      59721 non-null  float64
 14  instrumentalness  59721 non-null  float64
 15  liveness          59721 non-null  float64
 16  valence           59721 non-null  float64
 17  tempo   

In [42]:
df.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [43]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.443,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic
5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,False,0.688,0.481,6,-8.807,1,0.1050,0.2890,0.000000,0.1890,0.666,98.017,4,acoustic
7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,False,0.703,0.444,11,-9.331,1,0.0417,0.5590,0.000000,0.0973,0.712,150.960,4,acoustic


## Export Cleaned Dataset

In [44]:
OUTPUT_PATH = (PROJECT_ROOT / "cleaned_data.csv")

df.to_csv(OUTPUT_PATH,index=False)
OUTPUT_PATH

WindowsPath('c:/Users/Admin/OneDrive - National Economics University/Desktop/File final/Bước 2/cleaned_data.csv')